# Access fire data with a few functions

Place `fire_data.py` alongside this notebook. Install `numpy` and `matplotlib` in your environment, and download/extract the processed data separately. In Colab, upload the module and this notebook, then set the path below to your extracted split directory.

These examples use the **data/labels** archive format. Days start at **1** and mean stored observation positions, not calendar dates. Incomplete source tails are not included.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from fire_data import FireDataset

DATA_DIRECTORY = Path("./content/data/train")  # Change this for your machine or Colab.
dataset = FireDataset(DATA_DIRECTORY)
fire_ids = dataset.list_fires()
if not fire_ids:
    raise FileNotFoundError(f"No .npz fires found in {DATA_DIRECTORY}")
print(f"Found {len(fire_ids)} fires")
print(fire_ids[:10])


## Choose a fire and discover its bands
Choose an ID from the list above. Band names are case-insensitive.


In [ ]:
fire_id = fire_ids[0]  # Or use an ID such as "p20777152".
print(dataset.describe_fire(fire_id))
for channel, name in enumerate(dataset.list_bands(), start=1):
    print(f"{channel:2}: {name}")


## Get a whole fire, one day, or one band
The day axis comes first, so `len(fire)` gives the number of retained input observations.


In [ ]:
fire = dataset.get_fire(fire_id)                 # (days, bands, height, width)
day_number = 1
one_day = dataset.get_day(fire_id, day_number)   # (bands, height, width)
band_name = "I4 Night"
one_band = dataset.get_band(fire_id, band_name) # (days, height, width)
print("Whole fire:", fire.shape)
print("One day:", one_day.shape)
print("One band:", one_band.shape)
assert 1 <= day_number <= len(fire)
np.testing.assert_array_equal(one_day, fire[day_number - 1])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(one_band[day_number - 1], cmap="hot")
ax.set_title(f"{fire_id}: observation {day_number}, {band_name}")
ax.set_axis_off()
fig.colorbar(image, ax=ax, label="Stored processed value")
plt.show()


## Access prediction labels

Each window has one newly burned-area target for the following observation. Targets are at positions **11, 14, 17, ...**; these are not daily burned-area masks. The last target lies beyond the retained input timeline. Use the supplied positions when associating targets with observations.


In [ ]:
targets = dataset.get_labels(fire_id)
print("Mask shape:", targets["masks"].shape)
print("Target observation positions:", targets["target_days"])

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(targets["masks"][0], cmap="gray", vmin=0, vmax=1)
ax.set_title(f"Newly burned-area target: observation {targets['target_days'][0]}")
ax.set_axis_off()
plt.show()


## Check an actual archive

The helper rejects inconsistent overlaps instead of choosing conflicting values. This additional check compares the first window directly against the reconstructed fire and verifies target alignment. It does not certify the historical preprocessing or establish calendar dates.


In [ ]:
with np.load(DATA_DIRECTORY / f"{fire_id}.npz", allow_pickle=False) as archive:
    stored = archive["data"]
    np.testing.assert_array_equal(fire[:10], stored[0].transpose(1, 0, 2, 3))
    np.testing.assert_array_equal(targets["masks"], archive["labels"])
    assert len(fire) == 10 + 3 * (len(stored) - 1)
print("Archive shape, overlap, first-window values, and stored labels validated.")


## Practical notes

- Use a separate `FireDataset` for each train/validation/test directory.
- Values are already processed; thermal channels can contain cumulative maxima.
- Reading even one day initially decompresses a complete fire. Only the latest fire is cached, and returned arrays are independent copies.
- Unknown IDs, bands, and invalid day numbers produce errors with guidance.
- If overlap validation fails, verify the archive's preprocessing version and stride or regenerate it. Do not ignore the error.

Sources: [channel table](https://www.nature.com/articles/s41597-025-06271-3/tables/1) and [preprocessing notebook](https://colab.research.google.com/drive/1XLhx0HFi1Qgn8JNNN-fcfyh7WjdVZCEs).
